# VC+MVR Cascade HPR Targeting
This notebook demonstrates the new heat-pump-only `Vapour compression with MVR cascade` HPR backend. It shows the public `HPR_TYPE` configuration, solves the standalone MVR and combined VC+MVR unit models deterministically, and includes an optional full `PinchWorkspace` target solve.

## Case context
The packaged `heat_pump_targeting.json` case is a compact process heat-pump example. The VC+MVR backend adds a vapour-compression low stage whose condenser heat can either heat the process directly or drive one or more MVR high stages. MVR condenser heat is useful process heat; MVR evaporator duty is internal cascade heat in the combined model.

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display

from OpenPinch import PinchWorkspace
from OpenPinch.lib.enums import HPRcycle
from OpenPinch.services.heat_pump_integration.common.shared import (
    plot_multi_hp_profiles_from_results,
)
from OpenPinch.services.heat_pump_integration.unit_models import (
    MechanicalVapourRecompressionCycle,
    VapourCompressionMvrCascade,
)
from OpenPinch.utils import get_scalar_value

## Configure a public VC+MVR targeting case
The new backend is selected through the normal `HPR_TYPE` option. `N_COND` and `N_EVAP` define the low-stage vapour-compression placement dimensions; `N_MVR` defines the MVR high-stage count. `MVR_FLUIDS` is independent from the low-stage `REFRIGERANTS` list.

In [ ]:
workspace = PinchWorkspace(source="heat_pump_targeting.json", project_name="Plant")

carnot_case = workspace.copy_case("baseline", "reference_carnot", activate=False)
carnot_options = {
        "HPR_TYPE": HPRcycle.MultiTempCarnot.value,
        "HPR_LOAD_VALUE": 0.10,
        "MAX_HP_MULTISTART": 5,
        "N_COND": 1,
        "N_EVAP": 1,
}
carnot_case.update_options(carnot_options)

vc_mvr_case = workspace.copy_case("baseline", "vc_mvr_cascade", activate=False)
vc_mvr_options = {
        "HPR_TYPE": HPRcycle.VapourCompMVR.value,
        "HPR_LOAD_VALUE": 0.10,
        "MAX_HP_MULTISTART": 1,
        "N_COND": 1,
        "N_EVAP": 1,
        "N_MVR": 1,
        "REFRIGERANTS": "R245FA",
        "MVR_FLUIDS": "Water",
        "ETA_COMP": 0.75,
        "ETA_MVR_COMP": 0.72,
        "ETA_MOTOR": 0.95,
        "DT_HPR_CASCADE_HX": 5.0,
        "INITIALISE_SIMULATED_CYCLE": False,
        "BB_MINIMISER": "rbf_surrogate",
}
vc_mvr_case.update_options(vc_mvr_options)

pd.DataFrame(
    [
        {
            "case": "reference_carnot",
            "HPR_TYPE": carnot_options["HPR_TYPE"],
            "N_COND": carnot_options["N_COND"],
            "N_EVAP": carnot_options["N_EVAP"],
            "N_MVR": None,
        },
        {
            "case": "vc_mvr_cascade",
            "HPR_TYPE": vc_mvr_options["HPR_TYPE"],
            "N_COND": vc_mvr_options["N_COND"],
            "N_EVAP": vc_mvr_options["N_EVAP"],
            "N_MVR": vc_mvr_options["N_MVR"],
        },
    ]
)

## Standalone MVR thermodynamics
The MVR model accepts any CoolProp-compatible fluid and defaults to water. The inlet is saturated or superheated vapour at `T_evap`, the outlet is a real compressor discharge at condenser pressure, and the condenser outlet is saturated or subcooled liquid. `ETA_MOTOR` converts shaft work to electrical work.

In [ ]:
mvr = MechanicalVapourRecompressionCycle()
mvr.solve_from_source_heat(
    T_evap=80.0,
    T_cond=120.0,
    Q_source=500.0,
    dT_superheat=5.0,
    dT_subcool=4.0,
    eta_mvr_comp=0.72,
    eta_motor=0.95,
    fluid="Water",
)

pd.DataFrame(
    [
        {
            "fluid": mvr.refrigerant,
            "T_evap / degC": mvr.T_evap,
            "T_cond / degC": mvr.T_cond,
            "Q_evap / W": mvr.Q_evap,
            "Q_cond / W": mvr.Q_cond,
            "superheat / K": mvr.dT_superheat,
            "subcool / K": mvr.dT_subcool,
            "shaft work / W": mvr.shaft_work,
            "electric work / W": mvr.work,
            "COP_h": mvr.COP_h,
        }
    ]
)

## Combined VC+MVR cascade mechanics
The hottest VC condenser stage supplies source heat that generates saturated vapour for the first MVR stage. Each MVR stage dry-compresses source vapour, uses post-compression internal liquid injection to desuperheat the discharge, condenses a process split of the post-injection vapour, and passes the remaining vapour to the next stage. The external stream collection excludes the VC-to-MVR source heat by default because that transfer is internal to the cascade.

In [ ]:
cascade = VapourCompressionMvrCascade()
cascade.solve(
    T_evap_vc=np.array([20.0]),
    T_cond_vc=np.array([80.0]),
    dT_lift_mvr=np.array([15.0]),
    Q_heat_vc=np.array([1000.0]),
    mvr_source_fraction=0.45,
    dT_subcool_vc=np.array([0.0]),
    dT_subcool_mvr=np.array([4.0]),
    dT_ihx_gas_side_vc=np.array([0.0]),
    eta_comp=0.75,
    eta_mvr_comp=0.72,
    eta_motor=0.95,
    refrigerant=["Propane"],
    mvr_fluid=["Pentane"],
    dt_cascade_hx=5.0,
)

pd.DataFrame(
    {
        "stage": ["VC direct heat", "MVR condenser heat"],
        "external Q_heat / W": cascade.Q_heat_arr,
        "external Q_cool / W": cascade.Q_cool_arr,
        "work / W": cascade.work_arr,
    }
)

In [ ]:
external_streams = cascade.build_stream_collection(
    include_cond=True,
    include_evap=True,
    include_internal=False,
)
with_internal_streams = cascade.build_stream_collection(
    include_cond=True,
    include_evap=True,
    include_internal=True,
)

pd.DataFrame(
    [
        {
            "stream set": "external only",
            "stream count": len(external_streams.get_stream_names()),
            "total heat flow / W": external_streams.sum_stream_attribute("heat_flow"),
        },
        {
            "stream set": "including internal MVR evaporator",
            "stream count": len(with_internal_streams.get_stream_names()),
            "total heat flow / W": with_internal_streams.sum_stream_attribute("heat_flow"),
        },
    ]
)

## MVR and VC+MVR stream profiles
The deterministic unit-model stream collections can be sent through the same Plotly HPR profile helper used by the targeting backends. The standalone MVR view shows its source and condenser streams. The VC+MVR views compare the external stream collection with the optional internal VC-to-MVR source stream included.

In [ ]:
mvr_streams = mvr.build_stream_collection(include_cond=True, include_evap=True)
stream_rows = []
for label, streams in [
    ("Standalone MVR", mvr_streams),
    ("VC+MVR external", external_streams),
    ("VC+MVR with internal", with_internal_streams),
]:
    for stream in streams:
        stream_rows.append(
            {
                "collection": label,
                "stream": stream.name,
                "type": stream.type,
                "T_supply / degC": get_scalar_value(stream.t_supply),
                "T_target / degC": get_scalar_value(stream.t_target),
                "heat_flow": get_scalar_value(stream.heat_flow),
            }
        )

pd.DataFrame(stream_rows)


In [ ]:
display(
    plot_multi_hp_profiles_from_results(
        hpr_hot_streams=mvr_streams.get_hot_utility_streams(),
        hpr_cold_streams=mvr_streams.get_cold_utility_streams(),
        title="Standalone MVR source and condenser stream collection",
    )
)

display(
    plot_multi_hp_profiles_from_results(
        hpr_hot_streams=external_streams.get_hot_utility_streams(),
        hpr_cold_streams=external_streams.get_cold_utility_streams(),
        title="VC+MVR external stream collection",
    )
)

display(
    plot_multi_hp_profiles_from_results(
        hpr_hot_streams=with_internal_streams.get_hot_utility_streams(),
        hpr_cold_streams=with_internal_streams.get_cold_utility_streams(),
        title="VC+MVR stream collection including internal source heat",
    )
)


## Optional full target solve
The full VC+MVR target goes through the same public target surface as the other HPR backends. Simulated-cycle optimisation is slower and more sensitive to fluid feasibility than the Carnot targets, so this cell is disabled by default. Set `RUN_FULL_TARGET = True` when you want to run the optimiser-backed target.

In [ ]:
RUN_FULL_TARGET = False

if RUN_FULL_TARGET:
    rows = []
    for case_name in ["reference_carnot", "vc_mvr_cascade"]:
        problem = workspace.case(case_name)
        target = problem.target.direct_heat_pump()
        rows.append(
            {
                "case": case_name,
                "HPR_TYPE": problem.config.HPR_TYPE,
                "hpr_success": getattr(target, "hpr_success", None),
                "hpr_work": get_scalar_value(getattr(target, "hpr_work", None)),
                "hpr_cop": get_scalar_value(getattr(target, "hpr_cop", None)),
                "external_utility": get_scalar_value(
                    getattr(target, "hpr_external_utility", None)
                ),
            }
        )
    display(pd.DataFrame(rows))
else:
    print("Set RUN_FULL_TARGET = True to run the public optimiser-backed target.")

## Optional HPR plots
Once the full target solve has been run, the standard HPR-aware plots are available from the normal plot accessor.

In [ ]:
if RUN_FULL_TARGET:
    vc_mvr_case.target.direct_heat_pump()
    display(vc_mvr_case.plot.net_load_profiles(zone_name="Direct Heat Pump"))
    display(
        vc_mvr_case.plot.grand_composite_curve_with_heat_pump(
            zone_name="Direct Heat Pump"
        )
    )

## Interpretation
Use the deterministic unit-model cells to check thermodynamic accounting and internal heat routing before running a full placement solve. When moving to the public target, start with one VC stage and one MVR stage, keep `MAX_HP_MULTISTART` low while tuning fluids and load fraction, then increase stage counts and multistart effort once the temperature ranges are physically feasible.